In [5]:
import pandas as pd
import xgboost as xgb
import joblib
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

print("\n=======================================================")
print("📊 MODULE: BASELINE BENCHMARKING (ĐỘC LẬP)")
print("=======================================================\n")

# ---------------------------------------------------------
# Bước 1: LOAD DỮ LIỆU TỪ Ổ CỨNG
# ---------------------------------------------------------
print("Đang tải dữ liệu Vàng từ model_artifacts/ ...")
df_train = pd.read_parquet("D:/XAI_risk_score/data/gold/train_gold.parquet")
df_test = pd.read_parquet("D:/XAI_risk_score/data/gold/test_gold.parquet")
# (Logistic Regression không cần tập Validation để Early Stopping, nên ta chỉ cần Train và Test)

X_train = df_train.drop(columns=['target'])
y_train = df_train['target']

X_test = df_test.drop(columns=['target'])
y_test = df_test['target']

# ---------------------------------------------------------
# Bước 2: LOAD XGBOOST VÀ LẤY ĐIỂM AUC THỰC TẾ TRÊN TẬP TEST
# ---------------------------------------------------------
print("Đang tải mô hình XGBoost Core...")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("D:/XAI_risk_score/model_artifacts/xgboost_core_model.json")

# Dự đoán và tính điểm
y_test_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc_test = roc_auc_score(y_test, y_test_prob_xgb)

# ---------------------------------------------------------
# Bước 3: MỨC 1 - SINGLE FEATURE BASELINE
# ---------------------------------------------------------
top_feature = 'pctinstlsallpaidlat10d_839L'
sign = 1 if X_test[top_feature].corr(y_test) > 0 else -1

single_feature_prob = X_test[top_feature] * sign
single_feature_auc = roc_auc_score(y_test, single_feature_prob)

# ---------------------------------------------------------
# ---------------------------------------------------------
# Bước 4: MỨC 2 - HUẤN LUYỆN LOGISTIC REGRESSION BASELINE
# ---------------------------------------------------------
print("Đang chuẩn bị Pipeline cho Logistic Regression...")

# 1. Tự động nhận diện cột số và cột chữ
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f"-> Tìm thấy {len(numeric_cols)} cột số và {len(categorical_cols)} cột chữ.")

# 2. Tạo bộ tiền xử lý (Preprocessor)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        # handle_unknown='ignore' giúp tránh lỗi nếu tập Test có chữ lạ chưa từng gặp ở Train
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# 3. Nối vào Pipeline
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

print("Đang huấn luyện Logistic Regression (Sẽ mất thời gian hơn một chút do OneHotEncoding)...")
lr_pipeline.fit(X_train, y_train)

# Lưu mô hình Logistic Regression (Dùng joblib cho scikit-learn)
joblib.dump(lr_pipeline, "D:/XAI_risk_score/model_artifacts/lr_baseline_model.joblib")
print("✅ Đã lưu mô hình Baseline vào 'model_artifacts/lr_baseline_model.joblib'")

# Dự đoán trên tập Test
y_test_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]
lr_auc_test = roc_auc_score(y_test, y_test_prob_lr)

# ---------------------------------------------------------
# Bước 5: IN BÁO CÁO KẾT QUẢ SO SÁNH
# ---------------------------------------------------------
print("\n🏆 --- KẾT QUẢ BENCHMARK TRÊN TẬP TEST ---")
print(f"👉 1. Random Guess Baseline AUC      : 0.5000")
print(f"👉 2. Single Feature AUC ({top_feature[:15]}...) : {single_feature_auc:.4f}")
print(f"👉 3. Logistic Regression AUC        : {lr_auc_test:.4f}")
print(f"👑 4. XGBoost Core Model AUC         : {xgb_auc_test:.4f}")

improvement = (xgb_auc_test - lr_auc_test) / lr_auc_test * 100
print(f"\n💡 KẾT LUẬN CHO README:")
print(f"XGBoost cải thiện AUC thêm {improvement:.2f}% so với Logistic Regression Baseline.")

if improvement > 2.0:
    print("-> Đánh giá: Cải thiện ĐÁNG KỂ. XGBoost đã học được các tương tác phi tuyến (non-linear) mà LR không làm được.")
elif improvement > 0:
    print("-> Đánh giá: Cải thiện KHÔNG LỚN. Có thể cân nhắc đưa LR vào môi trường Production vì tính dễ giải thích của nó.")
else:
    print("-> Đánh giá: XGBoost THUA Logistic Regression! Cần kiểm tra lại hiện tượng Overfitting của XGBoost.")


📊 MODULE: BASELINE BENCHMARKING (ĐỘC LẬP)

Đang tải dữ liệu Vàng từ model_artifacts/ ...
Đang tải mô hình XGBoost Core...
Đang chuẩn bị Pipeline cho Logistic Regression...
-> Tìm thấy 21 cột số và 7 cột chữ.
Đang huấn luyện Logistic Regression (Sẽ mất thời gian hơn một chút do OneHotEncoding)...
✅ Đã lưu mô hình Baseline vào 'model_artifacts/lr_baseline_model.joblib'

🏆 --- KẾT QUẢ BENCHMARK TRÊN TẬP TEST ---
👉 1. Random Guess Baseline AUC      : 0.5000
👉 2. Single Feature AUC (pctinstlsallpai...) : 0.5749
👉 3. Logistic Regression AUC        : 0.7679
👑 4. XGBoost Core Model AUC         : 0.8046

💡 KẾT LUẬN CHO README:
XGBoost cải thiện AUC thêm 4.78% so với Logistic Regression Baseline.
-> Đánh giá: Cải thiện ĐÁNG KỂ. XGBoost đã học được các tương tác phi tuyến (non-linear) mà LR không làm được.
